# Задача от лекциите



In [ ]:
suppressMessages(library(tidyverse)

dt <- tibble(
  x = c(15, 6, 2, 19, 8),
  y = c(36, 25, 13, 49, 17)
)

dt)



Линейният регресионен модел за тази задача е (тъй като не е посочено нищо друго):

$$
y_i = \beta_1 + \beta_2 x_i + e_i, e_i \sim N(0, \sigma^2)
$$



In [ ]:
fit <- lm(y ~ 1 + x, data = dt)
summary(fit)



Методът на най-малките квадрати търси оценки $\hat{\beta}_1$ и $\hat{\beta}_2$ такива, че сумата на квадратите на разликите между прогнози и наблюдения да е възможно най-малка:

$$
\min_{\hat{\beta}_1, \hat{\beta}_2} RSS(\hat{\beta}_1, \hat{\beta}_2) = \sum_{i = 1}^{n}(y_i - \hat{y}_i)^2
$$

Прогнозите $\hat{y}_i$ се получават, когато в регресионното уравнение заместим неизвестните коефициенти $\beta_1$ и $\beta_2$ с техните оценки:

$$
\hat{y}_i = \hat{\beta}_1 + \hat{\beta}_2 x_i
$$

Решението на минимизационната задача е дадено от:

$$
\hat{\beta}_2 = \frac{\sum_{i = 1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i = 1}^n (x_i - \bar{x}) ^ 2}
$$

Пресмятаме средните стойности на $x$ и $y$ от данните: $\bar{x} = 10$ и $\bar{y} = 28$
Заместваме в тази формула и получаваме:

$$
\begin{align}
\hat{\beta}_2 & = \frac{(15 - 10)(36 - 28) + (6 - 10)(25 - 28) + (2 - 10)(13 - 28) + (19 - 10)(49 - 28) + (8 - 10)(17 - 28)}{(15 - 10)^2 + (6 - 10)^2 + (2 - 10)^2 + (19 - 10)^2 + (8 - 10)^2} \\
& \approx 2.016
\end{align}
$$

Оценката за $\beta_1$ получаваме от формулата:

$$
\hat{\beta}_1 = \bar{y} - \hat{\beta}_2 \bar{x} \\
\hat{\beta}_1 = 28 - 2.016 \cdot 10 = 7.84
$$

## Алтернативен начин за пресмятане на $\hat{\beta}_1$ и $\hat{\beta}_2$

Може да се покаже, че оценките за $\beta_1$ и $\beta_2$ са:

$$
\begin{align*}
\hat{\beta}_2 & = \frac{\overline{x y} - \bar{x} \bar{y}}{\overline{x^2} - \bar{x}^2} \\
\hat{\beta}_1 & = \bar{y} - \hat{\beta}_2 \bar{x}
\end{align*}
$$
За да приложите тази формула, пресметнете за всяко наблюдение $x_i y_i$, $x_i^2$ и $\bar{x}^2$:



In [ ]:
dt <- dt %>% 
  mutate(xy = x * y,
         x2 = x^2)
dt %>% knitr::kable()



Сега можем да пресметнем $\overline{x y}$, $\overline{x^2}$ и $\bar{x}^2$:



In [ ]:
dt %>%
  summarise(
    x_mean = mean(x),
    y_mean = mean(y),
    xy_mean = mean(xy),
    x2_mean = mean(x2),
    x_mean2 = mean(x)^2
  )




В следващата задача се търси $R^2$. Значението на $R^2$ се вижда от разлагането на общата вариация на $y$. Може да се покаже, че важи следната формула:

$$
\underbrace{\sum_{i = 1}^{n} (y_i - \bar{y})^2}_{TSS} = \underbrace{\sum_{i = 1}^{n}(y_i - \hat{y}_i)^2}_{RSS} + \underbrace{\sum_{i = 1}^{n} (\hat{y}_i - \bar{y})^2}_{ESS}
$$

$$
R^2 = 1 - \frac{RSS}{TSS}
$$

$$
TSS = (36 - 28)^2 + (25 - 28)^2 + (13 - 28)^2 + (49 - 28)^2 + (17 - 28)^2 = 860 \\
$$

За RSS трябва да изчислим остатъците $y_i - \hat{y}_i$. За първото наблюдение остатъкът е $36 - (7.842 + 2.016 \cdot 15) = -2.082$. За второто наблюдение остатъкът е $36 - (7.842 + 2.016 \cdot 6) = 5.06$. Другите остатъци можете да видите в следващата таблица в колонката `res`.




In [ ]:
dt <- dt %>%
  mutate(
    y_hat = 7.842 + 2.016 * x,
    res = y - y_hat
  )
dt



$$
RSS = (-2.08)^2 + 5.06^2 + 1.13^2 + 2.85^2 + (-6.97)^2 = 87.95
$$

От тук за $R^2$ получаваме:

$$
R^2 = 1 - \frac{87.95}{860} = 0.897
$$



In [ ]:
rho <- 2.016 / sum((dt$y - mean(dt$y)) ^ 2)



Следващата задача търси доверителен интервал за дисперсията $\sigma^2$ на $e_i$. От страница 28 в презентация 5 намираме, че


$$
(n - k) \frac{\hat{\sigma}^2}{\sigma^2} \sim \chi^2_{n - k}
$$


Чи-квадрат разпределението има един параметър, наречен степени на свобода (degrees of freedom). Степените на свобода в линейните регресионни модели са равни на броя наблюдения ($n = 5$) минус броя на коефициентите в модела (тук $k = 2$, тъй като има два коефициента: $\beta_1, \beta_2$).
me

За доверителния интервал (confidence interval, CI) в задачата е дадена вероятност за покритие от 95 процента. $1 - \alpha = 0.95 \implies \alpha = 0.05$. Знаем, че случайна променлива, която следва чи квадрат разпределение с $n - k$ степени на свобода ще се резлизира с верояност 95 процента между $\alpha / 2$-рия квантил ($\chi_{n - k}(\alpha/2)$) и $(1 - \alpha / 2)$ квантил: $\chi_{n - k}(1 - \alpha / 2)$. При $\alpha = 0.05$ търсим квантилите при $0.05 / 2 = 0.025$ и $1 - 0.05 / 2 = 0.975$ Тези квантили можем да пресметнем например в `R`:



In [ ]:
qchisq(0.025, df = 5 - 2)
qchisq(0.975, df = 5 - 2)



Когато трябва да използваме таблицата на страница 12 от презентация 4: първо намираме реда, съответстващ на степените на свобода: 3. Tаблицата в презентацията се различава от имплементацията в `R` и представя квантилите в обратен ред. На 0.975-тия квантил от обясненията тук в таблицата съответства 0.025-тия квантил на на стр. 13 (ред 3): 9.34. На 0.025-тия квантил от обясненията тук в таблицата съответства 0.975-тия квантил (стр. 12, ред 3: 0.215).

От дефиницията на квантилите получаваме, че вероятността $(n - k) \frac{\hat{\sigma}^2}{\sigma^2}$ да се реализира в интервала $[0.215; 9.34]$ е 95 процента.

$$
P\left( 0.215 < (n - k) \frac{\hat{\sigma}^2}{\sigma^2} < 9.34\right) = 0.95\\
$$

Опростяваме и преобразуваме двете неравенства, така че да получим интервал за $\sigma^2$. При умножаването със $\sigma^2$ използваме факта, че дисперсията е строго положителна, така че можем да умножим и двете страни на неравенство с нея без да променяме посоката на неравенството. Използваме също така, че в задачата е дадено $\hat{\sigma} = 5.41$.

$$
0.215 < 3 \frac{\hat{\sigma}^2}{\sigma^2} \implies \\
\frac{0.215}{3}\sigma^2 < \hat{\sigma}^2 \implies \\
\sigma^2 < \frac{3 \hat{\sigma}^2}{0.215} = \frac{3 \cdot 5.41 ^ 2}{0.215} \approx 408.4
$$

от второто неравенство получаваме

$$
3 \frac{\hat{\sigma}^2}{\sigma^2} < 9.34 \implies \\
3 \frac{\hat{\sigma}^2}{9.34} < \sigma^2 \implies \\
3 \frac{5.41^2}{9.34} < \sigma^2 \implies \\
9.4 < \sigma^2
$$

С това границите на 95 процентовия доверителен интервал за $\sigma^2$ са $[9.4; 408.4]$.


В последната задача се пита за тест на хипотезата относно един коефициент в модела. В условието не пише нищо за алтернатива, така че ще допуснем, че става въпрос за двустранна алтернатива.

$$
H_0: \beta_1 = 3 \\
H_1: \beta_1 \neq 3
$$

За този тест използваме следната статистика

$$
t = \frac{\hat{\beta}_1 - 3}{se(\hat{\beta}_1)}
$$

Заместваме (в условието имаме $se(\hat{\beta}_1) = 4.61$):

$$
t = \frac{7.84 - 3}{4.61} \approx 1.05
$$

Тази стойност сравняваме с квантилите на t разпределение с $n - k = 5 - 2 = 3$ степени на свобода (стр. 8 от презентация 4). От нивото на значимост (significance level) в задачата намираме кои квантили трябва да търсим:
$\alpha / 2$ и $(1 - \alpha / 2)$. При един процент ниво на значимост ($\alpha = 0.01$) трябва да намерим квантилите при $0.01 / 2 = 0.005$ и при $1 - 0.01 / 2 = 0.995$. Степените на свобода са същите (това не е случайно съвпадение) както и при чи-квадрат разпределението от примера с доверителния интервал: 3. Тъй като t разпределението е симетрично е достатъчно да намерим един от квантилите, другият е със същата абсолютна стойност и с обратен знак.

Търсения квантил намираме на ред 3 (df = 3) в таблицата на стр. 8 от презентация 4: 5.84



In [ ]:
qt(0.005, df = 5 - 2)



Следователно критичните стойности на теста са [-5.84; 5.84]. Сравняваме стойността на t статистиката с критичните стойности и виждаме, че тя е по-малка от тях по абсолютна стойност, следователно не можем да отхвърлим нулевата хипотеза при ниво на значимост 0.01.

В случай, че се получи стойност на t-статистиката по-голяма от горната критична стойност или по-ниска от долната критична стойност, тогава отхвърляме нулевата хипотеза при ниво на значимост 0.01.